# Save dataset to training, validation and test sets

In [1]:
# imports
from pathlib import Path
import plotly.graph_objects as go
import pandas as pd
import sys

# helpers
sys.path.insert(0, "../../")
from geometric_extraction_helper import (
    ALL_FEATURE_KEYS
)

from dataloader import (
    create_hf_dataset,
    show_label_distribution
)

/Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
# data directory and output path
DATA_DIR = "../3_5_clean_all_features/data_dropped_input_duplicates.parquet"
OUTPUT_PATH = "./data_original.parquet"
DATASET_NAME = "dataset"

# load parquet file
df = pd.read_parquet(DATA_DIR)
df.head()

,ifc_file_path,model_name,ifc_schema,guid,label_ifc_entity,label_predefined_type,storey,type_gks,eBKPh,excel_label_file_path,...,mat_kunststein,mat_kunststoff,mat_metall,mat_mörtel,mat_naturstein,mat_putz,mat_stahl,mat_zement,horizontal_elements_above,horizontal_elements_below
0,../../0_data/1_ifc_models/ZUST/ZUST_P_ARC_VOL_...,ZUST_P_ARC_VOL_,IFC4,1HyWKnRVf2nQPdSu_qBpKS,IfcSpace,GFA,U01,GF,"[C Konstruktion Gebäude, C05 Ergänzende Leistu...",../../0_data/1_ifc_models/ZUST/ZUST_Alle_ARC_A...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,../../0_data/1_ifc_models/ZUST/ZUST_P_ARC_VOL_...,ZUST_P_ARC_VOL_,IFC4,330umOG8j8lh$KH3zRhUgt,IfcSpace,GFA,U01,GF,"[C Konstruktion Gebäude, C05 Ergänzende Leistu...",../../0_data/1_ifc_models/ZUST/ZUST_Alle_ARC_A...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,../../0_data/1_ifc_models/ZUST/ZUST_P_ARC_VOL_...,ZUST_P_ARC_VOL_,IFC4,0RuGd4rvD2xeZxAp7YWBVl,IfcSpace,GFA,U01,GF,"[C Konstruktion Gebäude, C05 Ergänzende Leistu...",../../0_data/1_ifc_models/ZUST/ZUST_Alle_ARC_A...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,../../0_data/1_ifc_models/ZUST/ZUST_P_ARC_EBK_...,ZUST_P_ARC_EBK_,IFC4,0p4jVtekv8bero8khCpNIT,IfcColumn,NOTDEFINED,U01,"Beton, Stahlbeton Priorität 800 D300","[C03 Stützenkonstruktion, C03.02 Innenstütze]",../../0_data/1_ifc_models/ZUST/ZUST_Alle_ARC_A...,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0,-1.0
4,../../0_data/1_ifc_models/ZUST/ZUST_P_ARC_EBK_...,ZUST_P_ARC_EBK_,IFC4,2YC9riJTTCN89fdfN7wHz9,IfcColumn,NOTDEFINED,U01,"Beton, Stahlbeton D300","[C03 Stützenkonstruktion, C03.02 Innenstütze]",../../0_data/1_ifc_models/ZUST/ZUST_Alle_ARC_A...,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0,-1.0


## 1. Fill NaN values with the category "unknown" to label and minor label columns

In [3]:
# show main labels
label_cols = [c for c in df.columns if c.startswith("label_")]
label_cols

['label_ifc_entity',
 'label_predefined_type',
 'label_is_external',
 'label_load_bearing']

In [4]:
# fill NaN values with the category "unknown" (for some elements it is not important for the eBKP-H classification)
df[label_cols] = df[label_cols].fillna("unknown")

In [5]:
# show minor labels
minor_label_cols = [c for c in df.columns if c.startswith("minor_")]
minor_label_cols

['minor_label_unter_terrain',
 'minor_label_konstruktionsergaenzung',
 'minor_label_deckbelag',
 'minor_label_bekleidung',
 'minor_label_aussenliegendes_bauteil',
 'minor_label_erdverbunden',
 'minor_label_unterkonstruktion',
 'minor_label_verdunkelung',
 'minor_label_schutzschicht',
 'minor_label_sonnenschutz',
 'minor_label_einbau',
 'minor_label_aufzugstyp']

In [6]:
# fill NaN values with the category "irrelevant" (for some elements it is not important for the eBKP-H classification)
minor_label_cols
df[minor_label_cols] = df[minor_label_cols].fillna("unknown")

## 3. Prepare and split the data

In [7]:
# create dataset with all main and minor labels, stratified on both without removing rare classes
ds, project_info = create_hf_dataset(
    df, ALL_FEATURE_KEYS,
    target_cols=label_cols,
    minor_cols=minor_label_cols,
    label_cols=label_cols,
)

Count of projects in splits: 8 train | 2 val | 3 test
Projects in train split: ['ADEM' 'BRUT' 'CHLI' 'ESAK' 'GSRH' 'IMBU' 'KEHO' 'ZEGA']
Projects in val split: ['GERB' 'ZUST']
Projects in test split: ['KEPR' 'LUMU' 'RALU']

Count of elements in splits: 33,732 | 5,820 | 3,811
Leakage deduplication: 0 rows from validation ds removed, 0 rows from test ds removed.
Orphan removal: 2296 train | 99 val | 101 test rows removed.

Train: 31436 (72.5%) | Val: 5721 (13.2%) | Test: 3710 (8.6%)


## 4. Analyse the distribution of the classes in the splitted datasets

In [8]:
# show distribution of main labels after removing rare classes
show_label_distribution(ds, label_cols).head(n=60)

train  train_pct  validation  \
feature               value                                          
label_ifc_entity      IfcColumn         127        0.4          19   
                      IfcCovering       349        1.1         125   
                      IfcDoor           825        2.6         169   
                      IfcFurniture      171        0.5          98   
                      IfcPlate          954        3.0         230   
                      IfcRailing       7000       22.3         109   
                      IfcRoof           815        2.6         126   
                      IfcSlab          1632        5.2         313   
                      IfcSpace         3922       12.5         957   
                      IfcStairFlight    135        0.4          55   
                      IfcWall         14167       45.1        3170   
                      IfcWindow        1339        4.3         350   
label_predefined_type BASESLAB          219        0.7          92   
                      DOOR              822        2.6         167   
                      FLAT_ROOF         815        2.6         126   
                      FLOOR            1285        4.1         190   
                      FLOORING          349        1.1         125   
                      GATE                3        0.0           2   
                      GFA               230        0.7          51   
                      GUARDRAIL        7000       22.3         109   
                      INTERNAL         3692       11.7         906   
                      MOVABLE           123        0.4           4   
                      NOTDEFINED       1933        6.1        1605   
                      PARAPET           395        1.3          22   
                      PARTITIONING      795        2.5          93   
                      PLUMBINGWALL     1450        4.6         365   
                      ROOF              128        0.4          31   
                      SHEET             954        3.0         230   
                      SOLIDWALL        9904       31.5        1253   
                      WINDOW           1339        4.3         350   
label_is_external     false           10290       32.7        2185   
                      true            16918       53.8        2426   
                      unknown          4228       13.4        1110   
label_load_bearing    false            6554       20.8        1461   
                      true            11097       35.3        2395   
                      unknown         13785       43.9        1865   

                                      validation_pct  test  test_pct  
feature               value                                           
label_ifc_entity      IfcColumn                  0.3    45       1.2  
                      IfcCovering                2.2    37       1.0  
                      IfcDoor                    3.0   275       7.4  
                      IfcFurniture               1.7    17       0.5  
                      IfcPlate                   4.0    15       0.4  
                      IfcRailing                 1.9    42       1.1  
                      IfcRoof                    2.2    11       0.3  
                      IfcSlab                    5.5   213       5.7  
                      IfcSpace                  16.7   429      11.6  
                      IfcStairFlight             1.0    53       1.4  
                      IfcWall                   55.4  2238      60.3  
                      IfcWindow                  6.1   335       9.0  
label_predefined_type BASESLAB                   1.6    66       1.8  
                      DOOR                       2.9   271       7.3  
                      FLAT_ROOF                  2.2    11       0.3  
                      FLOOR                      3.3   132       3.6  
                      FLOORING                   2.2    37       1.0  
                      GATE  

In [9]:
# show distribution of main labels after removing rare classes
show_label_distribution(ds, label_cols).tail(n=6)

train  train_pct  validation  validation_pct  \
feature            value                                                   
label_is_external  false    10290       32.7        2185            38.2   
                   true     16918       53.8        2426            42.4   
                   unknown   4228       13.4        1110            19.4   
label_load_bearing false     6554       20.8        1461            25.5   
                   true     11097       35.3        2395            41.9   
                   unknown  13785       43.9        1865            32.6   

                            test  test_pct  
feature            value                    
label_is_external  false    2018      54.4  
                   true     1192      32.1  
                   unknown   500      13.5  
label_load_bearing false     670      18.1  
                   true     1449      39.1  
                   unknown  1591      42.9

## 5. Save the datasets

In [10]:
# save dataset as parquet files for without removed rare classes
for split in ["train", "validation", "test"]:
    ds[split].to_parquet(f"./{DATASET_NAME}_{split}.parquet")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]